# PartA_2

This notebook implements a simple neural network ensemble for the flower dataset.

Structure:
1. Utilities and data loading
2. ShallowVGGNet base learner
3. Ensemble benchmark and evaluation

##  Colab Setup

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")
# !unzip "/content/gdrive/MyDrive/Deep_learning_2/flower.h5.zip" -d "/content/gdrive/MyDrive/Deep_learning_2/"
!ls /content/gdrive/MyDrive/Deep_learning_2


## Utilities

This section contains the imports, the HDF5 loader, plotting utilities, and helper functions reused across all other notebook.

In [ ]:
import csv
import time
import h5py
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.callbacks import ModelCheckpoint

# source: https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ModelCheckpoint
# source: /Users/gobinfamily/Downloads/Tiny_ImageNet_Example_Lab_Solution.ipynb


In [ ]:
def loadDataH5():
    candidate_paths = [
        "data1.h5",
        "/content/data1.h5",
        "/content/gdrive/MyDrive/Deep_learning_2/data1.h5",
        "/content/drive/MyDrive/Deep_learning_2/data1.h5",
    ]

    data_path = None
    for candidate in candidate_paths:
        if os.path.exists(candidate):
            data_path = candidate
            break

    if data_path is None:
        raise FileNotFoundError(
            "Could not find data1.h5. Expected it in the current directory, "
            "/content, or /content/gdrive/MyDrive/Deep_learning_2."
        )

    print("Using data file:", data_path)

    with h5py.File(data_path, "r") as hf:
        trainX = np.array(hf.get("trainX"))
        trainY = np.array(hf.get("trainY"))
        valX = np.array(hf.get("valX"))
        valY = np.array(hf.get("valY"))

    print("trainX shape:", trainX.shape, "trainY shape:", trainY.shape)
    print("valX shape:", valX.shape, "valY shape:", valY.shape)
    return trainX, trainY, valX, valY


def plot_history(history, model_name, output_dir="plots_partA2"):
    os.makedirs(output_dir, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(history["accuracy"], label="train_accuracy")
    axes[0].plot(history["val_accuracy"], label="val_accuracy")
    axes[0].set_title(f"{model_name} Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(history["loss"], label="train_loss")
    axes[1].plot(history["val_loss"], label="val_loss")
    axes[1].set_title(f"{model_name} Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(
        os.path.join(output_dir, f"{model_name}_history.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


def plot_confusion_matrix(y_true, y_pred, model_name, class_names, output_dir="plots_partA2"):
    os.makedirs(output_dir, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred)

    row_sums = cm.sum(axis=1, keepdims=True)
    cm_normalized = np.divide(
        cm.astype("float"),
        row_sums,
        out=np.zeros_like(cm, dtype=float),
        where=row_sums != 0,
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=True)
    ax.set_title(f"{model_name} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(
        os.path.join(output_dir, f"{model_name}_confusion_matrix.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_normalized,
        display_labels=class_names,
    )
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=True, values_format=".2f")
    ax.set_title(f"{model_name} Normalized Confusion Matrix")
    plt.tight_layout()
    plt.savefig(
        os.path.join(output_dir, f"{model_name}_confusion_matrix_normalized.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

    np.savetxt(
        os.path.join(output_dir, f"{model_name}_confusion_matrix.csv"),
        cm,
        delimiter=",",
        fmt="%d",
    )
    np.savetxt(
        os.path.join(output_dir, f"{model_name}_confusion_matrix_normalized.csv"),
        cm_normalized,
        delimiter=",",
        fmt="%.6f",
    )

    return cm, cm_normalized


def save_classification_report(y_true, y_pred, class_names, model_name, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    report_text = classification_report(y_true, y_pred, target_names=class_names)
    report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

    txt_path = os.path.join(output_dir, f"{model_name}_classification_report.txt")
    with open(txt_path, "w") as report_file:
        report_file.write(report_text)

    csv_path = os.path.join(output_dir, f"{model_name}_classification_report.csv")
    with open(csv_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["label", "precision", "recall", "f1-score", "support"])
        for label, metrics in report_dict.items():
            if isinstance(metrics, dict):
                writer.writerow([
                    label,
                    metrics.get("precision"),
                    metrics.get("recall"),
                    metrics.get("f1-score"),
                    metrics.get("support"),
                ])


def plot_prediction_examples(
    images,
    y_true,
    y_pred,
    true_class,
    pred_class=None,
    max_images=6,
    model_name="model",
    output_dir="plots_partA2",
):
    os.makedirs(output_dir, exist_ok=True)

    if pred_class is None:
        indices = np.where((y_true == true_class) & (y_pred == true_class))[0]
        title = f"{model_name}: correctly classified class {true_class}"
        file_name = f"{model_name}_class_{true_class}_correct_examples.png"
    else:
        indices = np.where((y_true == true_class) & (y_pred == pred_class))[0]
        title = f"{model_name}: class {true_class} misclassified as class {pred_class}"
        file_name = f"{model_name}_class_{true_class}_pred_{pred_class}_examples.png"

    if len(indices) == 0:
        print(f"No matching examples found for {title}.")
        return

    indices = indices[:max_images]
    fig, axes = plt.subplots(1, len(indices), figsize=(3 * len(indices), 3))
    if len(indices) == 1:
        axes = [axes]

    for ax, idx in zip(axes, indices):
        ax.imshow(images[idx])
        ax.set_title(f"true={y_true[idx]}\npred={y_pred[idx]}")
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, file_name), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def get_top_confusion_pair(cm):
    confusion_only = cm.copy()
    np.fill_diagonal(confusion_only, 0)
    max_index = np.argmax(confusion_only)
    true_class, pred_class = np.unravel_index(max_index, confusion_only.shape)
    if confusion_only[true_class, pred_class] == 0:
        return None
    return true_class, pred_class


def plot_class_confusion_comparison(
    images,
    y_true,
    y_pred,
    true_class,
    pred_class,
    max_images=4,
    model_name="model",
    output_dir="plots_partA2",
):
    os.makedirs(output_dir, exist_ok=True)

    misclassified_idx = np.where((y_true == true_class) & (y_pred == pred_class))[0]
    true_correct_idx = np.where((y_true == true_class) & (y_pred == true_class))[0]
    pred_correct_idx = np.where((y_true == pred_class) & (y_pred == pred_class))[0]

    if len(misclassified_idx) == 0:
        print(
            f"No misclassified examples found for class {true_class} predicted as class {pred_class}."
        )
        return

    misclassified_idx = misclassified_idx[:max_images]
    true_correct_idx = true_correct_idx[:max_images]
    pred_correct_idx = pred_correct_idx[:max_images]

    columns = max(len(misclassified_idx), len(true_correct_idx), len(pred_correct_idx), 1)
    fig, axes = plt.subplots(3, columns, figsize=(3 * columns, 9))

    if columns == 1:
        axes = np.array(axes).reshape(3, 1)

    row_titles = [
        f"Misclassified: true={true_class}, pred={pred_class}",
        f"Correct examples of true class {true_class}",
        f"Correct examples of predicted class {pred_class}",
    ]
    row_indices = [misclassified_idx, true_correct_idx, pred_correct_idx]

    for row, (title, indices) in enumerate(zip(row_titles, row_indices)):
        for col in range(columns):
            ax = axes[row, col]
            if col < len(indices):
                idx = indices[col]
                ax.imshow(images[idx])
                ax.set_title(f"true={y_true[idx]}\npred={y_pred[idx]}")
                ax.axis("off")
            else:
                ax.axis("off")
        axes[row, 0].set_ylabel(title, rotation=90, fontsize=11, labelpad=20)

    plt.suptitle(
        f"{model_name}: visual comparison for confusion {true_class} -> {pred_class}",
        fontsize=14,
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            output_dir,
            f"{model_name}_class_{true_class}_vs_class_{pred_class}_comparison.png",
        ),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


def set_seed(seed):
    np.random.seed(seed)
    tf.random.set_seed(seed)


## Model

This section reuses the ShallowVGGNet architecture from the lecture/lab material as the common base learner for the ensemble.

In [ ]:
class ShallowVGGNet:

    @staticmethod
    def build(width, height, depth, classes):
        model = tf.keras.models.Sequential()
        model.add(keras.Input(shape=(height, width, depth)))

        # first CONV => CONV => POOL layer set
        model.add(tf.keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu"))
        model.add(tf.keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu"))
        model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

        # second CONV => CONV => POOL layer set
        model.add(tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu"))
        model.add(tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu"))
        model.add(keras.layers.MaxPooling2D(pool_size=(2, 2)))

        # first (and only) set of FC => RELU layers
        model.add(keras.layers.Flatten())
        model.add(keras.layers.Dense(512, activation="relu"))

        # softmax classifier
        model.add(keras.layers.Dense(classes, activation="softmax"))

        return model

## Benchmark

This section trains multiple independent ShallowVGGNet base learners with different random initializations, stores their validation probabilities, averages them, and evaluates the final ensemble.

In [ ]:
def train_single_ShallowVGGNet(
    learner_index,
    trainX,
    trainY,
    valX,
    valY,
    epochs=20,
    batch_size=32,
    output_dir="plots_partA2",
):
    model_name = f"baseLearner_{learner_index}"
    checkpoint_path = f"{model_name}_best.keras"

    set_seed(100 + learner_index)
    height, width, depth = trainX.shape[1:]
    num_classes = len(np.unique(trainY))
    model = ShallowVGGNet.build(
        width=width,
        height=height,
        depth=depth,
        classes=num_classes,
    )
    model.summary()

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    checkpoint_cb = ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    )

    train_start = time.perf_counter()
    history = model.fit(
        trainX,
        trainY,
        validation_data=(valX, valY),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[checkpoint_cb],
    )
    training_seconds = time.perf_counter() - train_start

    best_model = tf.keras.models.load_model(checkpoint_path)
    eval_start = time.perf_counter()
    val_loss, val_accuracy = best_model.evaluate(valX, valY, verbose=0)
    predict_start = time.perf_counter()
    val_probs = best_model.predict(valX, verbose=0)
    predict_seconds = time.perf_counter() - predict_start
    evaluation_seconds = time.perf_counter() - eval_start
    val_pred = np.argmax(val_probs, axis=1)

    print(f"\n{model_name} validation loss: {val_loss:.4f}")
    print(f"{model_name} validation accuracy: {val_accuracy:.4f}")
    print(f"{model_name} training time: {training_seconds:.2f}s")
    print(f"{model_name} prediction time: {predict_seconds:.4f}s ({(predict_seconds / len(valX)) * 1000:.3f} ms/image)")

    plot_history(history.history, model_name, output_dir=output_dir)

    return {
        "name": model_name,
        "history": history.history,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_probs": val_probs,
        "val_pred": val_pred,
        "training_seconds": training_seconds,
        "evaluation_seconds": evaluation_seconds,
        "predict_seconds": predict_seconds,
        "predict_ms_per_image": (predict_seconds / len(valX)) * 1000,
    }


def evaluate_prediction_set(valX, valY, y_pred, model_name, output_dir="plots_partA2"):
    class_names = [f"Class {index}" for index in range(17)]
    cm, cm_normalized = plot_confusion_matrix(
        valY, y_pred, model_name, class_names, output_dir=output_dir
    )

    print(f"\nClassification report for {model_name}:")
    print(classification_report(valY, y_pred, target_names=class_names))
    save_classification_report(valY, y_pred, class_names, model_name, output_dir)

    best_class = int(np.argmax(np.diag(cm_normalized)))
    best_class_score = np.diag(cm_normalized)[best_class]
    print(
        f"Best classified class for {model_name}: "
        f"class {best_class} with normalized recall {best_class_score:.2f}"
    )

    plot_prediction_examples(
        valX,
        valY,
        y_pred,
        true_class=best_class,
        pred_class=None,
        max_images=5,
        model_name=model_name,
        output_dir=output_dir,
    )

    top_confusion = get_top_confusion_pair(cm)
    if top_confusion is not None:
        true_class, pred_class = top_confusion
        plot_prediction_examples(
            valX,
            valY,
            y_pred,
            true_class=true_class,
            pred_class=pred_class,
            max_images=5,
            model_name=model_name,
            output_dir=output_dir,
        )
        plot_class_confusion_comparison(
            valX,
            valY,
            y_pred,
            true_class=true_class,
            pred_class=pred_class,
            max_images=4,
            model_name=model_name,
            output_dir=output_dir,
        )


def train_ensemble(trainX, trainY, valX, valY, n_learners=5, epochs=20, batch_size=32):
    output_dir = "plots_partA2"
    os.makedirs(output_dir, exist_ok=True)

    base_results = []
    all_probabilities = []

    for learner_index in range(1, n_learners + 1):
        print(f"\n{'=' * 60}")
        print(f"Training base learner {learner_index}/{n_learners}")
        print(f"{'=' * 60}")

        result = train_single_ShallowVGGNet(
            learner_index,
            trainX,
            trainY,
            valX,
            valY,
            epochs=epochs,
            batch_size=batch_size,
            output_dir=output_dir,
        )
        base_results.append(result)
        all_probabilities.append(result["val_probs"])

    # The ensemble prediction is formed by averaging the probabilities from all
    # ShallowVGGNet base learners, which follows the lecture and assignment appendix guidance.
    ensemble_start = time.perf_counter()
    ensemble_probs = np.mean(np.stack(all_probabilities, axis=0), axis=0)
    ensemble_pred = np.argmax(ensemble_probs, axis=1)
    ensemble_accuracy = np.mean(ensemble_pred == valY)

    num_classes = len(np.unique(trainY))
    one_hot_valY = tf.keras.utils.to_categorical(valY, num_classes=num_classes)
    ensemble_loss = tf.keras.losses.categorical_crossentropy(one_hot_valY, ensemble_probs)
    ensemble_loss = float(np.mean(ensemble_loss))
    ensemble_predict_seconds = time.perf_counter() - ensemble_start

    print(f"\nEnsemble validation loss: {ensemble_loss:.4f}")
    print(f"Ensemble validation accuracy: {ensemble_accuracy:.4f}")
    print(f"Ensemble aggregation/prediction time: {ensemble_predict_seconds:.4f}s ({(ensemble_predict_seconds / len(valY)) * 1000:.3f} ms/image)")

    evaluate_prediction_set(
        valX,
        valY,
        ensemble_pred,
        model_name="ensembleModel",
        output_dir=output_dir,
    )

    print("\nBase learner summary:")
    for result in base_results:
        print(
            f"{result['name']}: val_loss={result['val_loss']:.4f}, "
            f"val_accuracy={result['val_accuracy']:.4f}, "
            f"train_time={result['training_seconds']:.2f}s, "
            f"predict_time={result['predict_seconds']:.4f}s"
        )

    print(
        f"\nEnsembleModel: val_loss={ensemble_loss:.4f}, "
        f"val_accuracy={ensemble_accuracy:.4f}"
    )

    return base_results, {
        "ensemble_loss": ensemble_loss,
        "ensemble_accuracy": ensemble_accuracy,
        "ensemble_pred": ensemble_pred,
        "ensemble_probs": ensemble_probs,
    }


## Soft Voting Reference

The current ensemble corresponds to **equal-weight soft voting**: each base learner outputs SoftMax probabilities, these probabilities are averaged, and the final class is obtained with `argmax`.

This is aligned with the scikit-learn ensemble documentation on weighted average probabilities (soft voting). The current implementation uses the special case where all learners have equal weight.

In [ ]:
# source: https://scikit-learn.org/stable/modules/ensemble.html#weighted-average-probabilities-soft-voting

def soft_voting_from_probabilities(probability_list, weights=None):
    stacked = np.stack(probability_list, axis=0)
    if weights is None:
        averaged_probabilities = np.mean(stacked, axis=0)
    else:
        averaged_probabilities = np.average(stacked, axis=0, weights=weights)
    predicted_classes = np.argmax(averaged_probabilities, axis=1)
    return averaged_probabilities, predicted_classes


# Example usage after training:
# probability_list = [result["val_probs"] for result in base_results]
# averaged_probabilities, ensemble_pred = soft_voting_from_probabilities(probability_list)
# weighted_probabilities, weighted_pred = soft_voting_from_probabilities(
#     probability_list,
#     weights=[1, 1, 2, 1, 1],
# )

## Run the Ensemble

Execute the following cell to load the data and run the full benchmark.

In [ ]:
trainX, trainY, valX, valY = loadDataH5()
base_results, ensemble_results = train_ensemble(
    trainX,
    trainY,
    valX,
    valY,
    n_learners=5,
    epochs=20,
    batch_size=32,
)

## Export Results to CSV

Export the main metrics to CSV after running the benchmark.

In [ ]:
import csv
import os
import shutil
import time


def export_parta2_results(base_results, ensemble_results, output_dir="exports_partA2", copy_to_drive=True):
    os.makedirs(output_dir, exist_ok=True)

    base_summary_path = os.path.join(output_dir, "partA2_base_learners.csv")
    with open(base_summary_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["model_name", "val_loss", "val_accuracy", "training_seconds", "evaluation_seconds", "predict_seconds", "predict_ms_per_image"])
        for result in base_results:
            writer.writerow([
                result.get("name"),
                result.get("val_loss"),
                result.get("val_accuracy"),
                result.get("training_seconds"),
                result.get("evaluation_seconds"),
                result.get("predict_seconds"),
                result.get("predict_ms_per_image"),
            ])

    history_path = os.path.join(output_dir, "partA2_base_history.csv")
    with open(history_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["model_name", "epoch", "accuracy", "val_accuracy", "loss", "val_loss"])
        for result in base_results:
            history = result.get("history", {})
            epochs = len(history.get("accuracy", []))
            for epoch in range(epochs):
                writer.writerow([
                    result.get("name"),
                    epoch + 1,
                    history.get("accuracy", [None] * epochs)[epoch],
                    history.get("val_accuracy", [None] * epochs)[epoch],
                    history.get("loss", [None] * epochs)[epoch],
                    history.get("val_loss", [None] * epochs)[epoch],
                ])

    ensemble_path = os.path.join(output_dir, "partA2_ensemble_summary.csv")
    with open(ensemble_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["ensemble_loss", "ensemble_accuracy", "ensemble_predict_seconds", "ensemble_predict_ms_per_image"])
        writer.writerow([
            ensemble_results.get("ensemble_loss"),
            ensemble_results.get("ensemble_accuracy"),
            ensemble_results.get("ensemble_predict_seconds"),
            ensemble_results.get("ensemble_predict_ms_per_image"),
        ])

    master_path = os.path.join(output_dir, "partA2_master_summary.csv")
    with open(master_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["question", "model_name", "val_loss", "val_accuracy", "training_seconds", "predict_seconds", "predict_ms_per_image", "short_comment"])
        for result in base_results:
            writer.writerow(["PartA_2", result.get("name"), result.get("val_loss"), result.get("val_accuracy"), result.get("training_seconds"), result.get("predict_seconds"), result.get("predict_ms_per_image"), ""])
        writer.writerow(["PartA_2", "ensembleModel", ensemble_results.get("ensemble_loss"), ensemble_results.get("ensemble_accuracy"), "N/A", ensemble_results.get("ensemble_predict_seconds"), ensemble_results.get("ensemble_predict_ms_per_image"), ""])

    print(f"Saved CSV files to {output_dir}")
    print(base_summary_path)
    print(history_path)
    print(ensemble_path)

    if copy_to_drive:
        drive_targets = [
            "/content/gdrive/MyDrive/Deep_learning_2",
            "/content/drive/MyDrive/Deep_learning_2",
        ]
        copied = False
        for drive_dir in drive_targets:
            if os.path.isdir(drive_dir):
                drive_export_dir = os.path.join(drive_dir, output_dir)
                os.makedirs(drive_export_dir, exist_ok=True)
                shutil.copy2(base_summary_path, os.path.join(drive_export_dir, os.path.basename(base_summary_path)))
                shutil.copy2(history_path, os.path.join(drive_export_dir, os.path.basename(history_path)))
                shutil.copy2(ensemble_path, os.path.join(drive_export_dir, os.path.basename(ensemble_path)))
                print(f"Copied CSV files to {drive_export_dir}")
                plot_source_dir = "plots_partA2"
                for checkpoint_name in ["baseLearner_1_best.keras", "baseLearner_2_best.keras", "baseLearner_3_best.keras", "baseLearner_4_best.keras", "baseLearner_5_best.keras"]:
                    if os.path.isfile(checkpoint_name):
                        shutil.copy2(checkpoint_name, os.path.join(drive_dir, checkpoint_name))
                if os.path.isdir(plot_source_dir):
                    drive_plot_dir = os.path.join(drive_dir, plot_source_dir)
                    os.makedirs(drive_plot_dir, exist_ok=True)
                    for file_name in os.listdir(plot_source_dir):
                        source_file = os.path.join(plot_source_dir, file_name)
                        if os.path.isfile(source_file):
                            shutil.copy2(source_file, os.path.join(drive_plot_dir, file_name))
                    print(f"Copied plot files to {drive_plot_dir}")
                copied = True
                break
        if not copied:
            print("Google Drive export folder not found. CSV files were saved locally only.")


if "base_results" in globals() and "ensemble_results" in globals():
    export_parta2_results(base_results, ensemble_results)
else:
    print("Run the ensemble cell first, then rerun this export cell.")
